# 🚀 Wan2.1 快速启动 (stable-diffusion-art 方案)

**最简单可靠的方式**

1. 设置: Runtime → Change runtime type → **L4 GPU**
2. 运行: Runtime → Run all
3. 访问生成的链接

In [ ]:
# @title ▶️ 一键启动 Wan2.1
# @markdown ---
# @markdown **设置项:**

# @markdown 模型选择:
MODEL_SIZE = "1.3B"  # @param ["1.3B", "14B", "2.2"]

# @markdown 启用 ngrok 穿透 (需要 token):
USE_NGROK = False  # @param{type:"boolean"}
NGROK_TOKEN = ""  # @param{type:"string"}

# ============== 执行代码 ==============
import os
import subprocess
import time
import requests
from IPython.display import clear_output, display, HTML

# 检查GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.free', '--format=csv,noheader'], 
                       capture_output=True, text=True)
gpu_info = result.stdout.strip()
print(f"🖥️  {gpu_info}")

# 设置路径
WORKSPACE = "/workspace"
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)

# 克隆 ComfyUI
if not os.path.exists(f"{WORKSPACE}/ComfyUI"):
    print("📦 克隆 ComfyUI...")
    subprocess.run(['git', 'clone', 'https://github.com/comfyanonymous/ComfyUI.git'], 
                  stdout=subprocess.DEVNULL)
    
    # 安装依赖
    print("📦 安装依赖...")
    subprocess.run(['pip', 'install', '-r', 'requirements.txt'], 
                  cwd=f"{WORKSPACE}/ComfyUI", stdout=subprocess.DEVNULL)

print("✅ 环境准备完成")

In [ ]:
# @title 📥 下载 Wan2.1 模型
# @markdown ---

import os

MODELS_DIR = f"{WORKSPACE}/ComfyUI/models/checkpoints"
os.makedirs(MODELS_DIR, exist_ok=True)

# 模型链接 (示例 - 需要替换为实际可用链接)
MODEL_URLS = {
    "1.3B": "https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B/resolve/main/Wan2.1_T2V_1.3B_bf16.safetensors",
    "14B": "https://huggingface.co/Wan-AI/Wan2.1-T2V-14B/resolve/main/Wan2.1_T2V_14B_bf16.safetensors",
    "2.2": "https://huggingface.co/Wan-AI/Wan2.2-T2V/resolve/main/Wan2.2_T2V.safetensors",
}

model_file = f"wan2.1_t2v_{MODEL_SIZE.lower()}.safetensors"
model_path = f"{MODELS_DIR}/{model_file}"

if not os.path.exists(model_path):
    print(f"📥 下载 Wan2.1 {MODEL_SIZE}...")
    print(f"   目标: {model_file}")
    print(f"   这可能需要几分钟时间...")
    
    # 使用 wget 下载 (可续传)
    result = subprocess.run([
        'wget', '-c', '-O', model_path, 
        MODEL_URLS[MODEL_SIZE]
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✅ 模型下载完成: {model_file}")
    else:
        print(f"⚠️  下载失败，请手动下载模型")
else:
    print(f"✅ 模型已存在: {model_file}")

# 列出已下载模型
print("\n📁 已下载模型:")
os.system(f"ls -lh {MODELS_DIR}")

In [ ]:
# @title 🚀 启动 ComfyUI
# @markdown ---

import subprocess
import threading
import time
from pyngrok import ngrok

# 启动 ComfyUI
def start_comfyui():
    os.chdir(f"{WORKSPACE}/ComfyUI")
    subprocess.Popen(
        ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        preexec_fn=os.setsid
    )

# 启动线程
start_thread = threading.Thread(target=start_comfyui)
start_thread.start()

print("⏳ ComfyUI 启动中 (约30秒)...")
time.sleep(30)

# ngrok 穿透
if USE_NGROK and NGROK_TOKEN:
    print("🔗 设置 ngrok 穿透...")
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8188, "http")
    display(HTML(f"""
    <div style='padding: 20px; background: linear-gradient(135deg, #1a1a2e, #16213e); 
                border-radius: 12px; margin: 20px 0;'>
        <h2 style='color: #00d4ff;'>🌐 访问地址</h2>
        <a href='{public_url}' target='_blank' 
           style='font-size: 24px; color: #00d4ff; text-decoration: none;'>
           {public_url}
        </a>
        <p style='color: #888;'>点击打开 ComfyUI 界面</p>
    </div>
"""))
else:
    # 使用 localtunnel
    print("🔗 使用 localtunnel 穿透...")
    subprocess.Popen(["npx", "localtunnel", "--port", "8188"], 
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    display(HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #1a1a2e, #16213e); 
                border-radius: 12px; margin: 20px 0;'>
        <h2 style='color: #00d4ff;'>🌐 访问说明</h2>
        <p>1. 点击下方链接打开 tunnel</p>
        <p>2. 在新页面点击 "Click to Continue"</p>
        <p>3. 等待 ComfyUI 界面加载</p>
        <p style='color: #ff6b6b;'>⚠️  每次运行都会生成新的链接</p>
    </div>
"""))
    !npx localtunnel --port 8188
else:
    # 本地访问
    display(HTML(f"""
    <div style='padding: 20px; background: #1a1a2e; border-radius: 12px; margin: 20px 0;'>
        <h2 style='color: #00d4ff;'>🔵 本地访问</h2>
        <p>直接在 Colab 中访问:</p>
        <a href='http://localhost:8188' target='_blank'>http://localhost:8188</a>
    </div>
"""))

---

## 📝 使用方法

1. 访问上方生成的链接
2. 导入工作流 (可选): 
   - 下载 [Wan2.1 工作流 JSON](https://raw.githubusercontent.com/theelderemo/wan2.2-google-colab/main/workflows/wan2.2-t2v.json)
   - 拖入 ComfyUI 界面
3. 输入提示词，点击 Queue Prompt

## ⚠️ 注意事项
- 免费 Colab 有 12 小时会话限制
- 30 分钟无操作会断开
- 建议测试完成后及时保存结果